# Trabalho em grupo – A frota nova da Horizonte
**Métodos e Ferramentas de Data Science – FGV MBA** · Em grupo (até 3 alunos), entrega individual · 30% da nota final

## A história
Quinta-feira, depois da reunião. A Marina, gerente de frota da **Horizonte Distribuidora** (empresa fictícia do NB01), gostou da análise do etanol e voltou com uma pergunta maior:

> *"A diretoria aprovou uma filial nova, com carros 0 km para os vendedores. Tenho três SUVs do mesmo porte na mesa: um **a combustão**, um **híbrido** e um **elétrico**. Antes de assinar o pedido, quero saber **quanto cada um custa no primeiro ano** e **se a diferença de preço do elétrico se paga**. Os vendedores de lá rodam bastante; a quilometragem está no briefing."*

O seu grupo é a equipe de análise. O sorteio diz **o estado da filial** e **quantos km cada carro roda por ano**.

## O que a Marina vai receber
Cada parte responde a uma pergunta da Marina:

| Parte | A pergunta da Marina | O que entra na nota |
|---|---|---|
| 1 | Os dados | – |
| 2 | Etanol ou gasolina no carro a combustão? | P1 |
| 3 | Quanto custa a energia de um ano? | P2, P3, P4 |
| 4 | O IPVA muda a conta? Quanto custa o mais barato? | P5, P6 |
| 5 | A diferença de preço do elétrico se paga? | P7 |
| 6 | E se a conta de luz tiver imposto? | P8 |
| 7 | Duas perguntas sobre as Aulas 2 e 3 | P9, P10 |
| 8 | A resposta para a Marina: o gráfico e o e-mail | E-mail |

**Nota:** P1 a P10 valem 0,8 ponto cada (8 pontos) e o e-mail para a Marina vale 2 pontos. A base é o que vimos no NB01 e na NB01a (filtros, `groupby`, `mean`, coluna nova, gráfico). O que for novo está explicado no bloco **"O que acontece na célula abaixo"**, antes de cada célula de código: leia primeiro, execute depois. As lacunas estão marcadas com `____` e `SEU CÓDIGO AQUI`.

**Tempo estimado:** de 4 a 6 horas para o grupo, começando na Aula 3. **Prazo:** 23h59 da véspera da Aula 4 (data no ECLASS).

## Premissas
- Primeiro ano de um carro **0 km** comprado na capital do estado.
- Consumo **combinado**: 55% cidade e 45% estrada, o peso usual da etiqueta do Inmetro.
- Carro flex abastece com o combustível **mais barato por km** (etanol ou gasolina).
- Elétrico carregado em casa, pela tarifa residencial **sem tributos** (a parte 6 testa o efeito dos tributos).

## Como entregar (cada aluno entrega)
1. Abra no Colab e clique em *Arquivo → Salvar uma cópia no Drive*.
2. Preencha **a sua matrícula** e **as matrículas do grupo** (todos do grupo usam a mesma lista). Execute o sorteio.
3. Complete as partes 2 a 8. Cada resposta vira uma variável; a seção RESPOSTAS junta tudo.
4. Execute o **verificador de formato** e corrija o que ele apontar.
5. Baixe o notebook **já executado** (*Arquivo → Fazer download → Baixar o .ipynb*), renomeie para `TG_<sua matrícula>.ipynb`, com **a sua própria matrícula** (não a que sorteou o estado), e envie no ECLASS.

## Seu grupo
Digite **a sua matrícula** (o seu nome de usuário do ECLASS, por exemplo `A12345678`) e **a lista de matrículas do grupo**, incluindo a sua. Todos do grupo usam **exatamente a mesma lista**.

In [ ]:
MATRICULA = ""   # SEU CÓDIGO AQUI: a sua matrícula, por exemplo "A12345678"
MEMBROS = ["", "", ""]   # SEU CÓDIGO AQUI: as matrículas do grupo, incluindo a sua (deixe "" onde não houver)

## O briefing (sorteio)
Não altere esta célula. O estado da filial e a quilometragem vêm da **menor matrícula** da lista, então todos do grupo recebem o mesmo briefing.

> **Resultado esperado:** o grupo, o estado da filial e quantos km cada carro roda por ano (15, 20, 25 ou 30 mil). Se aparecer `AssertionError`, confira a matrícula e a lista.

In [ ]:
import re
# Só estados em que as fontes do IPVA concordam (sem "conferir" na tabela de IPVA)
ESTADOS = ["AC", "AM", "BA", "DF", "GO", "MG", "MS", "MT", "PA", "PB", "PE", "PR", "RJ", "RN", "RS", "SC", "SP", "TO"]
NOMES = {"AC": "Acre", "AM": "Amazonas", "BA": "Bahia", "DF": "Distrito Federal", "GO": "Goiás", "MG": "Minas Gerais",
         "MS": "Mato Grosso do Sul", "MT": "Mato Grosso", "PA": "Pará", "PB": "Paraíba", "PE": "Pernambuco",
         "PR": "Paraná", "RJ": "Rio de Janeiro", "RN": "Rio Grande do Norte", "RS": "Rio Grande do Sul",
         "SC": "Santa Catarina", "SP": "São Paulo", "TO": "Tocantins"}
KMS = [15000, 20000, 25000, 30000]

def numero(matricula):
    return int(re.sub(r"\D", "", matricula) or 0)

membros = sorted({m.strip().upper() for m in MEMBROS if m.strip()})
assert 1 <= len(membros) <= 3, "O grupo tem de 1 a 3 matrículas. Confira a lista MEMBROS."
assert MATRICULA.strip().upper() in membros, "A sua MATRICULA precisa estar na lista MEMBROS."
n = numero(min(membros, key=numero))
UF = ESTADOS[n % len(ESTADOS)]
KM_POR_ANO = KMS[(n // len(ESTADOS)) % len(KMS)]
print("Grupo:", ", ".join(membros))
print(f"Filial da Horizonte: {UF} ({NOMES[UF]})")
print(f"Cada carro da filial roda {KM_POR_ANO:,} km por ano".replace(",", "."))

## Parte 1 – Os dados
Agora são **quatro tabelas** públicas, não uma:

| Tabela | O que traz | Fonte |
|---|---|---|
| `anp` | Preço do etanol e da gasolina nos postos, 1º semestre de 2026 (a mesma do NB01) | [ANP – Série histórica de preços](https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/serie-historica-de-precos-de-combustiveis) |
| `tarifa` | Tarifa residencial de energia (TE + TUSD, **sem tributos**) da distribuidora da capital, vigente em 13/10/2026 | [ANEEL – Tarifas homologadas](https://dadosabertos.aneel.gov.br/dataset/tarifas-distribuidoras-energia-eletrica) |
| `carros` | Consumo dos três SUVs e valor FIPE de outubro/2026 | Inmetro, PBE Veicular 2026; FIPE |
| `ipva` | Alíquota de IPVA 2026 do 1º ano, por estado e tecnologia | Legislação estadual compilada de fontes secundárias (`dados/README.md`) |

**O que acontece na célula abaixo**

- `BASE` guarda o começo do endereço, comum aos quatro arquivos; `BASE + "nome.csv"` junta os dois textos.
- Cada `pd.read_csv` cria uma tabela: `anp`, `tarifa`, `carros` e `ipva`.
- `carros.set_index("tecnologia")` faz a coluna `tecnologia` virar o **índice** (o nome das linhas). Assim, depois, buscamos um carro pelo nome: `carros.loc["Híbrido", "valor_fipe"]`.
- As colunas de consumo da etiqueta estão em **km por litro**, na cidade e na estrada. As duas linhas seguintes criam o **consumo combinado**: 0,55 × cidade + 0,45 × estrada, para os três carros de uma vez.
- `mj_km` é o consumo de energia em **megajoules por km**, que a etiqueta informa para todos os carros. É o que usamos no elétrico. `valor_fipe` é o preço de referência, base do IPVA.

**Resultado esperado:** `anp: (422418, 8)` e uma tabela com os três carros. O elétrico fica `NaN` (vazio) nas colunas de litros, porque não usa combustível. Repare que o híbrido faz mais km por litro que o carro a combustão.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://raw.githubusercontent.com/carloscarlim/fgv-mfds/main/dados/"
anp = pd.read_csv(BASE + "anp_combustiveis_2026s1.csv.gz")
tarifa = pd.read_csv(BASE + "aneel_tarifa_residencial_2026.csv")
carros = pd.read_csv(BASE + "inmetro_pbev_2026_suv.csv")
ipva = pd.read_csv(BASE + "ipva_2026_uf.csv")
print("anp:", anp.shape)

carros = carros.set_index("tecnologia")
carros["etanol_kml"] = 0.55 * carros["etanol_cidade"] + 0.45 * carros["etanol_estrada"]
carros["gasolina_kml"] = 0.55 * carros["gasolina_cidade"] + 0.45 * carros["gasolina_estrada"]
carros[["modelo", "etanol_kml", "gasolina_kml", "mj_km", "valor_fipe"]]

## Parte 2 – Etanol ou gasolina?
A primeira coisa que a Marina vai perguntar: *"o carro a combustão da filial roda com quê?"*

No NB01 usamos a regra dos 70%, que é uma aproximação. Aqui fazemos a conta exata, com o consumo **deste** carro:

**custo por km = preço do litro ÷ km por litro**

Vale o combustível com o **menor** custo por km.

**Pergunta 1.** No seu estado, o carro a combustão deve abastecer com `"etanol"` ou `"gasolina"`?

**O que acontece na célula abaixo**

Esta célula tem **duas lacunas**.

- `estado = anp[anp["uf"] == UF]` filtra as coletas do estado sorteado, como no NB01.
- A linha do `groupby("produto")` calcula o preço médio de cada produto no estado. O resultado `precos` é uma lista com nome em cada posição (no pandas, uma **Series**): `precos["ETANOL"]` é o preço médio do etanol.
- `carros.loc["Combustão", "etanol_kml"]` busca na tabela `carros` a **linha** `Combustão` e a **coluna** `etanol_kml`, como um endereço de célula do Excel.
- `custo_km_etanol` já está pronto: preço do etanol ÷ km por litro com etanol (R\$ por litro ÷ km por litro = R\$ por km).
- **Lacuna 1:** faça o mesmo para a gasolina.
- **Lacuna 2:** guarde em `p1_combustivel` o texto `"etanol"` ou `"gasolina"`, o de menor custo por km. Pode escrever o texto direto, olhando o resultado, ou usar `"etanol" if custo_km_etanol < custo_km_gasolina else "gasolina"`, que escolhe sozinho.

**Resultado esperado:** os preços médios do etanol e da gasolina no estado e o custo por km de cada um, entre R\$ 0,50 e R\$ 0,66.

In [ ]:
estado = anp[anp["uf"] == UF]
precos = estado[estado["produto"].isin(["ETANOL", "GASOLINA"])].groupby("produto")["preco_venda"].mean()
print(precos.round(2))

custo_km_etanol = precos["ETANOL"] / carros.loc["Combustão", "etanol_kml"]
custo_km_gasolina = ____     # SEU CÓDIGO AQUI: preço da gasolina ÷ km por litro de gasolina do carro a combustão
print(f"Etanol: R$ {custo_km_etanol:.3f}/km | Gasolina: R$ {custo_km_gasolina:.3f}/km")

p1_combustivel = ____        # SEU CÓDIGO AQUI: "etanol" ou "gasolina", o de menor custo por km

## Parte 3 – Quanto custa rodar um ano?
*"Quanto vou gastar de energia com cada carro?"*

- Carros **flex** (combustão e híbrido): custo por km = o **menor** entre etanol e gasolina.
- **Elétrico**: custo por km = **kWh por km × tarifa**. A etiqueta dá o consumo em MJ por km; como 1 kWh = 3,6 MJ, dividir por 3,6 converte para kWh por km.
- Custo de um ano = custo por km × `KM_POR_ANO`, a quilometragem do briefing.

**Pergunta 2.** Custo de energia de um ano do carro **a combustão** (R\$).
**Pergunta 3.** Custo de energia de um ano do **híbrido** (R\$).
**Pergunta 4.** Custo de energia de um ano do **elétrico**, com a tarifa sem tributos (R\$).

**O que acontece na célula abaixo**

Esta célula tem **quatro lacunas**. O carro a combustão está pronto e serve de modelo.

- `min(a, b)` devolve o **menor** de dois números: é o motorista escolhendo o combustível mais barato por km. Multiplicado por `KM_POR_ANO`, vira o custo de um ano.
- **Híbrido:** repita as contas da parte 2, agora com `carros.loc["Híbrido", ...]`, e depois o `min(...) * KM_POR_ANO`.
- **Elétrico:** `kwh_km` já converte MJ para kWh. `tarifa.set_index("uf").loc[UF, "tarifa_kwh_sem_tributos"]` busca a tarifa do estado (R\$ por kWh). Falta multiplicar kWh por km × tarifa × `KM_POR_ANO`.
- No `print`, o `:,.0f` mostra o número sem casas decimais e com separador de milhar (no padrão americano, com vírgula).

**Resultado esperado:** o elétrico gasta perto de um quarto do que gasta o carro a combustão; o híbrido fica no meio, cerca de 20% abaixo do carro a combustão.

In [ ]:
# Combustão (exemplo pronto)
p2_energia_combustao = min(custo_km_etanol, custo_km_gasolina) * KM_POR_ANO

# Híbrido
hib_etanol = ____            # SEU CÓDIGO AQUI: preço do etanol ÷ km por litro de etanol do híbrido
hib_gasolina = ____          # SEU CÓDIGO AQUI: o mesmo para a gasolina
p3_energia_hibrido = ____    # SEU CÓDIGO AQUI: o menor custo por km × KM_POR_ANO

# Elétrico
kwh_km = carros.loc["Elétrico", "mj_km"] / 3.6
tarifa_uf = tarifa.set_index("uf").loc[UF, "tarifa_kwh_sem_tributos"]
p4_energia_eletrico = ____   # SEU CÓDIGO AQUI: kWh por km × tarifa × KM_POR_ANO

print(f"Combustão: R$ {p2_energia_combustao:,.0f} | Híbrido: R$ {p3_energia_hibrido:,.0f} | Elétrico: R$ {p4_energia_eletrico:,.0f}")

## Parte 4 – O imposto que muda a conta
*"Somando o IPVA do primeiro ano, quanto custa cada carro, e qual sai mais barato?"* Vários estados isentam o elétrico e o híbrido; outros cobram de todos.

**IPVA = alíquota ÷ 100 × valor FIPE** do carro.

**Pergunta 5.** IPVA do primeiro ano do **elétrico** no seu estado (R\$). Pode ser zero.
**Pergunta 6.** Somando energia e IPVA, qual é o **custo do primeiro ano do carro mais barato** (R\$)?

Anote também **qual** é o carro mais barato: ele vai para o e-mail da Marina.

**O que acontece na célula abaixo**

Esta célula tem **quatro lacunas**.

- `aliq = ipva.set_index("uf").loc[UF]` pega a **linha inteira** do estado na tabela de IPVA: `aliq["aliquota_hibrido"]` é a alíquota do híbrido, em %.
- `pd.Series({...})` monta uma lista com nome em cada posição, a partir de um **dicionário** (`{"nome": valor, ...}`). A primeira posição, a do carro a combustão, está pronta.
- `total = energia + ipva_carro` soma as duas listas **posição a posição** (combustão com combustão, híbrido com híbrido).
- O `pd.DataFrame({...})` só junta as três listas numa tabela para mostrar.
- `total.min()` devolve o **menor valor**; `total.idxmin()` devolve o **nome** da posição com o menor valor (o carro).

**Resultado esperado:** a observação do IPVA do estado, uma tabela de 3 linhas (os carros) com energia, IPVA e total, e o nome do carro mais barato.

In [ ]:
aliq = ipva.set_index("uf").loc[UF]
print("Observação do IPVA:", aliq["observacao"])

ipva_carro = pd.Series({
    "Combustão": aliq["aliquota_combustao"] / 100 * carros.loc["Combustão", "valor_fipe"],
    "Híbrido": ____,         # SEU CÓDIGO AQUI: alíquota do híbrido ÷ 100 × valor FIPE do híbrido
    "Elétrico": ____,        # SEU CÓDIGO AQUI: o mesmo para o elétrico
})
energia = pd.Series({"Combustão": p2_energia_combustao, "Híbrido": p3_energia_hibrido, "Elétrico": p4_energia_eletrico})
total = energia + ipva_carro
print(pd.DataFrame({"energia": energia, "ipva": ipva_carro, "total": total}).round(0))

p5_ipva_eletrico = ____      # SEU CÓDIGO AQUI: o IPVA do elétrico
p6_total_mais_barato = ____  # SEU CÓDIGO AQUI: o menor total
print("Mais barato:", total.idxmin())

## Parte 5 – A diferença de preço se paga?
O elétrico custa mais caro que o carro a combustão na compra. A Marina pergunta: *"em quanto tempo a economia paga essa diferença?"* É o **payback simples**:

**payback (anos) = diferença de preço ÷ economia de um ano**

**Pergunta 7.** Payback simples do **elétrico** em relação ao carro **a combustão**, em anos.

**O que acontece na célula abaixo**

Esta célula tem **duas lacunas**.

- `diferenca_preco` já está pronta: valor FIPE do elétrico menos o do carro a combustão.
- **Lacuna 1:** a economia de um ano é o total do carro a combustão **menos** o total do elétrico, ambos da série `total` da parte 4 (por exemplo, `total["Combustão"]`).
- **Lacuna 2:** o payback é a diferença de preço ÷ a economia de um ano.

**Resultado esperado:** um número entre 0,5 e 3 anos.

In [ ]:
diferenca_preco = carros.loc["Elétrico", "valor_fipe"] - carros.loc["Combustão", "valor_fipe"]
economia_ano = ____          # SEU CÓDIGO AQUI: total da combustão menos total do elétrico
p7_payback = ____            # SEU CÓDIGO AQUI: diferença de preço ÷ economia de um ano
print(f"Diferença de preço: R$ {diferenca_preco:,.0f} | Economia no 1º ano: R$ {economia_ano:,.0f} | Payback: {p7_payback:.1f} anos")

## Parte 6 – E se a conta de luz tiver imposto?
Alguém do financeiro vai lembrar: *"a tarifa da ANEEL vem sem tributos, e o preço da bomba já tem imposto; a comparação favorece o elétrico"*. Antes da reunião, teste o efeito: suponha que os tributos deixem a conta de luz **30% mais cara** (um valor hipotético; o ICMS e o PIS/Cofins variam por estado).

**Pergunta 8.** Com a energia elétrica 30% mais cara, qual é o **total do primeiro ano do elétrico** (energia + IPVA), em R\$?

Depois, compare com os outros dois carros: o mais barato mudou? Isso vai para o e-mail da parte 8.

**O que acontece na célula abaixo**

Esta célula tem **uma lacuna**.

- `FATOR_TRIBUTOS = 1.3` quer dizer 30% a mais.
- **Lacuna:** só a energia do elétrico fica mais cara: energia do elétrico × `FATOR_TRIBUTOS`, mais o IPVA do elétrico (que não muda).
- O último `print` mostra os totais do carro a combustão e do híbrido, para comparar.

**Resultado esperado:** um total do elétrico maior que o da parte 4. Em quase todos os briefings, o carro mais barato continua o mesmo; em um deles, muda.

In [ ]:
FATOR_TRIBUTOS = 1.3
p8_total_eletrico_tributos = ____   # SEU CÓDIGO AQUI: energia do elétrico × FATOR_TRIBUTOS + IPVA do elétrico
print(f"Total do elétrico com tributos: R$ {p8_total_eletrico_tributos:,.0f}")
print(f"Combustão: R$ {total['Combustão']:,.0f} | Híbrido: R$ {total['Híbrido']:,.0f}")

## Parte 7 – Duas perguntas sobre as Aulas 2 e 3
**Pergunta 9.** A Marina quer dividir os vendedores da filial em **perfis de uso** (quem roda muito em estrada, quem roda pouco na cidade, e assim por diante) a partir dos registros de quilometragem, **sem ter os perfis definidos antes**. Que técnica é essa?
Responda com uma destas palavras: `"classificacao"`, `"previsao"`, `"agrupamento"`, `"reducao"` ou `"anomalia"`.

**Pergunta 10.** Qual é a principal limitação do payback da parte 5?
- `"a"`: usa o preço médio da capital, e não o do estado.
- `"b"`: o consumo do elétrico foi medido em km por litro.
- `"c"`: usa só o primeiro ano; se uma isenção de IPVA acabar, a economia dos anos seguintes muda.
- `"d"`: o valor FIPE já inclui o IPVA do primeiro ano.

In [ ]:
p9_tecnica = ""        # SEU CÓDIGO AQUI
p10_limitacao = ""     # SEU CÓDIGO AQUI: "a", "b", "c" ou "d"

## Parte 8 – A resposta para a Marina
A análise só vale se chegar a quem decide. A Marina vai receber um gráfico e um e-mail curto.

1. Faça **um gráfico de barras** com o total do primeiro ano de cada carro (a série `total`), com título e nome no eixo. *Dica:* `total.plot(kind="bar")`, como os gráficos do NB01.
2. Escreva o **e-mail para a Marina**, em até 4 frases (vale 2 pontos, 0,5 por item):
   - **a recomendação**, com o número que a sustenta;
   - **o que muda** se a conta de luz tiver tributos (parte 6);
   - **o cuidado** antes de assinar o pedido;
   - **clareza:** escrito para quem decide, sem código e sem jargão.

In [ ]:
# SEU CÓDIGO AQUI: gráfico de barras de `total`, com título e nome no eixo

**O que acontece na célula abaixo**

Esta célula guarda o e-mail num **texto** (`str`). As três aspas (`"""`) permitem escrever em várias linhas.

- Escreva o e-mail **entre as três aspas**, no lugar de `Marina, ...`. Pode usar acentos e quebrar linhas à vontade.
- Use os números das partes anteriores: o Python não completa o texto sozinho.

**Resultado esperado:** o `print` mostra o seu e-mail.

In [ ]:
EMAIL_MARINA = """
Marina, ...
"""
print(EMAIL_MARINA)

## RESPOSTAS
Não altere esta célula: ela junta as respostas das partes 2 a 8.

In [ ]:
RESPOSTAS = {
    "P1_combustivel": p1_combustivel,
    "P2_energia_combustao": p2_energia_combustao,
    "P3_energia_hibrido": p3_energia_hibrido,
    "P4_energia_eletrico": p4_energia_eletrico,
    "P5_ipva_eletrico": p5_ipva_eletrico,
    "P6_total_mais_barato": p6_total_mais_barato,
    "P7_payback": p7_payback,
    "P8_total_eletrico_tributos": p8_total_eletrico_tributos,
    "P9_tecnica": p9_tecnica,
    "P10_limitacao": p10_limitacao,
    "EMAIL": EMAIL_MARINA,
}

## Verificador de formato
Execute antes de baixar o notebook. Ele confere o formato (não a resposta certa) e mostra um resumo das suas respostas: valores em reais sem centavos e o payback com uma casa decimal.

In [ ]:
import json
import unicodedata

def sem_acento(t):
    return unicodedata.normalize("NFKD", str(t)).encode("ascii", "ignore").decode().strip().lower()

def verificar(r):
    ok = True
    numeros = {"P2_energia_combustao": 0, "P3_energia_hibrido": 0, "P4_energia_eletrico": 0,
               "P5_ipva_eletrico": 0, "P6_total_mais_barato": 0, "P7_payback": 1, "P8_total_eletrico_tributos": 0}
    saida = {}
    for chave, casas in numeros.items():
        try:
            v = float(r.get(chave))
        except (TypeError, ValueError):
            print(f"⚠️ {chave}: precisa ser um número"); ok = False; continue
        saida[chave] = round(v, casas) if casas else int(round(v))
    saida["P1_combustivel"] = sem_acento(r.get("P1_combustivel", ""))
    if saida["P1_combustivel"] not in ("etanol", "gasolina"):
        print("⚠️ P1_combustivel: use etanol ou gasolina"); ok = False
    if r.get("P9_tecnica") not in ("classificacao", "previsao", "agrupamento", "reducao", "anomalia"):
        print("⚠️ P9_tecnica: use classificacao, previsao, agrupamento, reducao ou anomalia"); ok = False
    if r.get("P10_limitacao") not in ("a", "b", "c", "d"):
        print("⚠️ P10_limitacao: use a, b, c ou d"); ok = False
    saida["P9_tecnica"], saida["P10_limitacao"] = r.get("P9_tecnica"), r.get("P10_limitacao")
    email = str(r.get("EMAIL", "")).strip()
    if len(email) < 80 or email.startswith("Marina, ..."):
        print("⚠️ EMAIL: escreva o e-mail para a Marina na parte 8"); ok = False
    print()
    print("Resumo das respostas do grupo:")
    print(f"  Sua matrícula: {MATRICULA.strip().upper()}")
    print(f"  Matrículas do grupo: {', '.join(membros)}")
    print(f"  Briefing: {UF}, {KM_POR_ANO} km por ano")
    for chave in sorted(saida, key=lambda k: int(k.split('_')[0][1:])):
        valor = saida[chave]
        if isinstance(valor, float):
            valor = f"{valor:.1f}".replace(".", ",")
        print(f"  {chave.split('_')[0]}: {valor}")
    print(f"  E-mail: {len(email.split())} palavras")
    print()
    print("✅ Formato OK. Pode baixar e entregar." if ok else "Corrija os itens acima antes de entregar.")
    print("RESPOSTAS_JSON=" + json.dumps({"MATRICULA": MATRICULA.strip().upper(), "MEMBROS": membros, "UF": UF,
                                         "KM_POR_ANO": KM_POR_ANO, **saida, "EMAIL": email}, ensure_ascii=False, default=str))

verificar(RESPOSTAS)

## Erros comuns

| Mensagem | O que aconteceu | O que fazer |
|---|---|---|
| `AssertionError: A sua MATRICULA precisa estar na lista MEMBROS` | A sua matrícula não está na lista do grupo | Inclua a sua matrícula em `MEMBROS` |
| `NameError: name 'UF' is not defined` | O sorteio não foi executado | Execute as células desde o início (*Ambiente de execução → Executar anteriores*) |
| `NameError: name '____' is not defined` | Ficou uma lacuna sem preencher | Troque `____` pelo seu código |
| `KeyError: 'Híbrido'` | A célula da parte 1 rodou duas vezes (o `set_index` já tinha sido aplicado) | *Ambiente de execução → Reiniciar e executar tudo* |
| `KeyError: 'ETANOL'` | O filtro de produto ficou vazio | Confira as maiúsculas: `"ETANOL"` e `"GASOLINA"` |
| `TypeError` no verificador | Uma resposta ficou como texto vazio `""` | Complete a pergunta indicada |

## Plano B
Se o download falhar (rede da empresa), baixe os quatro arquivos da pasta `dados/` do repositório da disciplina, carregue-os pelo painel de arquivos do Colab (ícone de pasta, à esquerda) e troque a linha `BASE = ...` por `BASE = ""`.

**Fontes dos dados:** ANP (preços dos combustíveis, 1º semestre de 2026), ANEEL (tarifa residencial), Inmetro/Conpet (PBE Veicular 2026), FIPE (valor de outubro/2026) e IPVA 2026 compilado de fontes secundárias. Detalhes em `dados/README.md`. A Horizonte Distribuidora e a Marina são fictícias.

### Autor e licença

**Autor:** Prof. Carlos Carlim · [LinkedIn](https://www.linkedin.com/in/carloscarlim-77144120) · [GitHub](https://github.com/carloscarlim)

**Licença MIT.** Copyright (c) 2026 Carlos Carlim. Os direitos autorais deste material pertencem exclusivamente ao autor.

É permitido usar, copiar, modificar, mesclar, publicar, distribuir, sublicenciar e vender cópias deste material, desde que este aviso de copyright e o texto da licença sejam mantidos em todas as cópias ou partes substanciais. O material é fornecido "no estado em que se encontra", sem garantia de qualquer tipo. Texto completo da licença: [LICENSE](https://github.com/carloscarlim/fgv-mfds/blob/main/LICENSE).

Os dados usados nos notebooks têm licenças próprias, indicadas em [`dados/README.md`](https://github.com/carloscarlim/fgv-mfds/blob/main/dados/README.md).